# S04 — Polynomial Regression and Regularisation
Campus practical · LO3 and LO7

This notebook generates deterministic synthetic nonlinear data. Compare polynomial degrees and Ridge/Lasso using identical development folds. Keep the final test set untouched until selection is complete. Synthetic results are for learning only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
rng = np.random.default_rng(SEED)
X = np.sort(rng.uniform(-3, 3, 500)).reshape(-1, 1)
y = 4 + 1.5*X[:, 0] - 1.2*X[:, 0]**2 + 0.35*X[:, 0]**3 + rng.normal(0, 3, 500)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print('scikit-learn:', sklearn.__version__)
print('Development:', len(X_dev), 'Reserved test:', len(X_test))

## 1. Inspect the data
One row is one independent synthetic observation. The input is available at prediction time; the target is numeric. The generator includes curvature and noise by design.

In [ ]:
plt.scatter(X_dev[:, 0], y_dev, alpha=0.6)
plt.xlabel('Input x'); plt.ylabel('Target y'); plt.title('Synthetic development data')
plt.show()

## 2. Build the fold-local pipeline
Polynomial expansion and scaling are fitted separately within each training fold.

In [ ]:
def make_model(degree=1, penalty='none', alpha=1.0):
    if penalty == 'ridge': estimator = Ridge(alpha=alpha)
    elif penalty == 'lasso': estimator = Lasso(alpha=alpha, max_iter=20000)
    else: estimator = LinearRegression()
    return Pipeline([('poly', PolynomialFeatures(degree=degree, include_bias=False)),
                     ('scale', StandardScaler()), ('model', estimator)])

def cv_summary(model):
    scores = cross_validate(model, X_dev, y_dev, cv=cv,
        scoring={'mae':'neg_mean_absolute_error','mse':'neg_mean_squared_error'},
        return_train_score=True)
    return {'train_mae': -scores['train_mae'].mean(),
            'validation_mae': -scores['test_mae'].mean(),
            'validation_sd': scores['test_mae'].std(ddof=1),
            'validation_rmse': np.sqrt(-scores['test_mse']).mean()}

for d in [1, 3]:
    print(d, cv_summary(make_model(degree=d)))

## 3. Student task — compare polynomial degrees
Complete the loop and record train MAE, validation MAE and fold standard deviation for degrees 1, 2, 3, 5 and 8. Explain any training/validation gap.

In [ ]:
degree_results = []  # TODO
for degree in [1, 2, 3, 5, 8]:
    # TODO: append {'degree': degree, **cv_summary(make_model(degree=degree))}
    pass
assert len(degree_results) == 5, 'Record all five degrees.'
display(pd.DataFrame(degree_results))

## 4. Student task — Ridge and Lasso
Compare at least two alpha values for each penalty, holding degree and folds fixed. Explain why scaling matters.

In [ ]:
regularisation_results = []  # TODO
for penalty in ['ridge', 'lasso']:
    for alpha in [0.01, 0.1, 1.0, 10.0]:
        # TODO: append {'penalty': penalty, 'alpha': alpha, **cv_summary(make_model(degree=5, penalty=penalty, alpha=alpha))}
        pass
assert len(regularisation_results) == 8, 'Record four alpha values for both penalties.'
display(pd.DataFrame(regularisation_results))

## 5. Lock your selection before testing
Choose the candidate using development CV only. Write the chosen degree, penalty and alpha below before running the final test cell.

In [ ]:
selected_degree = None  # TODO: set from your experiment
selected_penalty = None  # 'none', 'ridge' or 'lasso'
selected_alpha = 1.0
assert selected_degree is not None and selected_penalty in ['none','ridge','lasso'], 'Record selection first.'

## 6. Final test — run once after selection is locked
Refit on all development rows, then evaluate the reserved test. Do not use these results to revise degree or alpha.

In [ ]:
final_model = make_model(selected_degree, selected_penalty, selected_alpha)
final_model.fit(X_dev, y_dev)
test_prediction = final_model.predict(X_test)
print('Test MAE:', mean_absolute_error(y_test, test_prediction))
print('Test RMSE:', np.sqrt(mean_squared_error(y_test, test_prediction)))
print('Test R2:', r2_score(y_test, test_prediction))

## 7. Residual diagnostic and exit ticket
Residual = actual minus predicted. The test plot is descriptive only; do not use it to iterate on the model.

In [ ]:
residuals = y_test - test_prediction
plt.scatter(test_prediction, residuals, alpha=0.65)
plt.axhline(0, linestyle='--')
plt.xlabel('Predicted target'); plt.ylabel('Residual (actual - predicted)')
plt.title('Final test residual diagnostic'); plt.show()

Answer individually: Why can a high-degree polynomial overfit? How do Ridge and Lasso differ? Why scale inside each fold? What evidence selected your model? Why must the test set not select the model?

Save your private notebook copy and experiment log.